1: Importació de llibreries (inclosos KerasTuner i AutoKeras)

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model, save_model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# Llibreries específiques per al Bonus
import keras_tuner as kt
import autokeras as ak

2: Càrrega i preparació de les dades bàsiques (Subset Òptim)

In [2]:
# Ruta local que has indicat
DATA_PATH = r"C:\Users\Alumne_mati1\Documents\machine_learning_course_cifo_2026\data\data_original\bank_term_deposit\bank_term_deposit.csv"

df = pd.read_csv(DATA_PATH)

# 1. Eliminar dades del leaderboard
df_labeled = df[df['split'] == 'labeled'].copy()

# 2. Selecció de les features recomanades de l'exercici anterior per evitar soroll
FEATURES_OPTIMAS = ['duration', 'balance', 'age', 'day', 'campaign']
X = df_labeled[FEATURES_OPTIMAS]
y = df_labeled['y'].map({'yes': 1, 'no': 0}).values

# 3. Divisió Train / Validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Preprocessament amb Pipeline
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), categorical_features)
    ])

X_train_p = preprocessor.fit_transform(X_train)
X_val_p = preprocessor.transform(X_val)
input_dim = X_train_p.shape[1]

3: Entrenar, Desar (save_model) i Carregar (load_model) amb Early Stopping

In [3]:
# 1. Definir un model base alternatiu
model_base = Sequential([
    Input(shape=(input_dim,)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_base.compile(optimizer=Adam(learning_rate=0.01), loss='binary_crossentropy', metrics=['accuracy'])

# 2. Callback d'Early Stopping
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# 3. Entrenament complet
model_base.fit(X_train_p, y_train, validation_data=(X_val_p, y_val), epochs=100, callbacks=[early_stop], verbose=0)

# 4. GUARDAR EL MODEL (Bonus: save_model)
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)
model_path = os.path.join(MODEL_DIR, "bank_model_base.keras")
save_model(model_base, model_path)
print(f"Model desat correctament a: {model_path}")

# 5. CARREGAR EL MODEL (Bonus: load_model)
model_carregat = load_model(model_path)
loss, acc = model_carregat.evaluate(X_val_p, y_val, verbose=0)
print(f"Model carregat i verificat amb èxit. Val Accuracy: {acc:.4f}")

Model desat correctament a: models\bank_model_base.keras
Model carregat i verificat amb èxit. Val Accuracy: 0.9428


4: Hyperparameter Tuning amb KerasTuner

In [4]:
# 1. Definir la funció constructor del model per a KerasTuner
def build_hypermodel(hp):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    
    # Tunar el nombre de neurones de la primera capa oculta (entre 16 i 64)
    hp_units = hp.Int('units', min_value=16, max_value=64, step=16)
    model.add(Dense(units=hp_units, activation='relu'))
    
    # Tunar si apliquem Dropout o no per evitar overfitting
    if hp.Boolean("dropout"):
        model.add(Dropout(rate=0.2))
        
    model.add(Dense(1, activation='sigmoid'))
    
    # Tunar el learning rate de l'optimizador Adam
    hp_lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    model.compile(optimizer=Adam(learning_rate=hp_lr), loss='binary_crossentropy', metrics=['accuracy'])
    return model

# 2. Configurar el cercador (fent servir RandomSearch)
tuner = kt.RandomSearch(
    build_hypermodel,
    objective='val_accuracy',
    max_trials=5, # Executarà 5 combinacions aleatòries diferents
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='bank_tuning'
)

# 3. Executar la cerca d'hiperparàmetres amb Early Stopping intermedi
tuner.search(X_train_p, y_train, epochs=30, validation_data=(X_val_p, y_val), callbacks=[EarlyStopping(monitor='val_loss', patience=5)])

# 4. Obtenir els millors resultats
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("\n--- Millors Hiperparàmetres Trobats ---")
print(f"Neurones optimitzades: {best_hps.get('units')}")
print(f"Afegir capa dropout?: {best_hps.get('dropout')}")
print(f"Learning rate optimitzat: {best_hps.get('learning_rate')}")

Reloading Tuner from keras_tuner_dir\bank_tuning\tuner0.json

--- Millors Hiperparàmetres Trobats ---
Neurones optimitzades: 32
Afegir capa dropout?: True
Learning rate optimitzat: 0.01


5: AutoKeras (Cerca de la millor arquitectura directament des de les dades reals)

In [7]:
# 1. Preparar las matrices de NumPy limpias
X_train_np = X_train.to_numpy().astype(np.float32)
X_val_np = X_val.to_numpy().astype(np.float32)

y_train_np = y_train.astype(np.int64)
y_val_np = y_val.astype(np.int64)

# 2. Definir un constructor flexible que busque la mejor ARQUITECTURA de red
def build_dynamic_architecture(hp):
    model = Sequential()
    model.add(Input(shape=(input_dim,)))
    
    # BUSCAR EL NÚMERO DE CAPAS OCULTAS (Entre 1 y 3 capas)
    for i in range(hp.Int('num_layers', min_value=1, max_value=3)):
        # BUSCAR LAS NEURONAS DE CADA CAPA (Entre 16 i 64)
        model.add(Dense(
            units=hp.Int(f'units_{i}', min_value=16, max_value=64, step=16),
            activation='relu'
        ))
        
        # BUSCAR SI CADA CAPA LLEVA DROPOUT
        if hp.Boolean(f'dropout_{i}'):
            model.add(Dropout(rate=0.2))
            
    # Capa de salida fija de clasificación binaria
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(
        optimizer=Adam(learning_rate=0.001), 
        loss='binary_crossentropy', 
        metrics=['accuracy']
    )
    return model

# 3. Configurar el buscador de arquitecturas
architecture_tuner = kt.RandomSearch(
    build_dynamic_architecture,
    objective='val_accuracy',
    max_trials=5,
    overwrite=True,
    directory='keras_tuner_architecture',
    project_name='bank_arch_search'
)

# 4. Iniciar la búsqueda automática de la red
print("Buscando la mejor arquitectura de red (Capas y Neuronas)...")
architecture_tuner.search(
    X_train_np, y_train_np, 
    epochs=20, 
    validation_data=(X_val_np, y_val_np),
    callbacks=[EarlyStopping(monitor='val_loss', patience=3)]
)

# 5. Extraer y mostrar los resultados de la mejor estructura encontrada
best_architecture = architecture_tuner.get_best_hyperparameters(num_trials=1)[0]
print("\n--- Millor Arquitectura Trobada ---")
print(f"Número total de capas ocultas: {best_architecture.get('num_layers')}")
for i in range(best_architecture.get('num_layers')):
    print(f" -> Capa {i+1}: {best_architecture.get(f'units_{i}')} neurones | Dropout: {best_architecture.get(f'dropout_{i}')}")

# 6. Exportar el mejor modelo entrenado (CAMBIADO num_trials POR num_models AQUÍ)
best_model = architecture_tuner.get_best_models(num_models=1)[0]
best_model.summary()

# 7. GUARDAR EL MEJOR MODELO ENCONTRADO POR KERASTUNER
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)
best_model_path = os.path.join(MODEL_DIR, "bank_best_architecture.keras")

# Guardamos el modelo optimizado
save_model(best_model, best_model_path)
print(f"¡Modelo óptimo de KerasTuner guardado con éxito en: {best_model_path}!")

# 8. CARREGAR I VERIFICAR EL MODEL GUARDAT
loaded_best_model = load_model(best_model_path)
loss, acc = loaded_best_model.evaluate(X_val_np, y_val_np, verbose=0)
print(f"Modelo cargado correctamente. Precisión final de validación comprobada: {acc:.4f}")

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.927860677242279

Best val_accuracy So Far: 0.927860677242279
Total elapsed time: 00h 00m 14s

--- Millor Arquitectura Trobada ---
Número total de capas ocultas: 1
 -> Capa 1: 16 neurones | Dropout: False


c:\Users\Alumne_mati1\Documents\machine_learning_course_cifo_2026\.venv_py312\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            96 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 113 (452.00 B)

 Trainable params: 113 (452.00 B)

 Non-trainable params: 0 (0.00 B)

c:\Users\Alumne_mati1\Documents\machine_learning_course_cifo_2026\.venv_py312\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'adam', because it has 10 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


¡Modelo óptimo de KerasTuner guardado con éxito en: models\bank_best_architecture.keras!
Modelo cargado correctamente. Precisión final de validación comprobada: 0.9279
